In [2]:
import os

import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [50]:
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [8]:
# Path 현재 경로를 쉽게 가져올 수 있음
SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

In [7]:
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
재택근무는 주 2회까지 가능하다.

제3조 (휴가)
연차휴가는 근로기준법에 따라 부여한다.
경조사 휴가는 별도 규정에 따른다.
자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

주요 트렌드
RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

주요 기능
음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
자동 스케줄: 시간대별 기기 자동 제어
에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
보안 모드: 외출 시 자동 보안 설정
"""
}

In [9]:
for filename, content in sample_texts.items():
    (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
    # sample_data/product_manual.txt

In [12]:
csv_data = [
    {'이름': '김철수', '부서': '개발팀', '직급': '선임'},
    {'이름': '이영희', '부서': '기획팀', '직급': '매니저'},
    {'이름': '박지민', '부서': '개발팀', '직급': '주임'}
]

with open(SAMPLE_DIR / 'employees.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['이름', '부서', '직급'])
    writer.writeheader()
    writer.writerows(csv_data)

In [14]:
# RAG : 검색해서 증강해서 확장한다 Retrieval Augmented Generation (내부 검색 엔진을 둔다)
# Fine-tuning과 비교해서 RAG 단점 : 우리가 생각하는 것과는 조금 다른 결과 값이 나올 가능성이 있음

# Hallucination(환각) : 생성형 모델들이 토큰단위로 생성
# 검색을 해서 잘 생성을 하게 하기 위한 내용

# Fine-tuning : 새롭게 학습을 시키는 것, GPT를 새로 교육을 시킨다 (비용과 시간이 오래 걸림)

In [51]:
knowledge_base = [
    {'id': 1, 'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'source': 'programming_guide.txt'},
    {'id': 2, 'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'source': 'ai_glossary.txt'},
    {'id': 3, 'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'source': 'database_manual.txt'},
]

query = 'RAG 기술이 무엇인가요?'

def retrieve_by_similarity(query, documents):
    contents = [doc['content'] for doc in documents]

    query_vec = embeddings.embed_query(query)
    doc_vec = embeddings.embed_documents(contents)

    scores = cosine_similarity([query_vec], doc_vec)[0]

    ranked = sorted(zip(documents, scores), key=lambda x:x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked]

retrieve_by_similarity(query, knowledge_base)

[({'id': 2,
   'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.',
   'source': 'ai_glossary.txt'},
  0.5587391524470275),
 ({'id': 1,
   'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.',
   'source': 'programming_guide.txt'},
  0.2584505348581131),
 ({'id': 3,
   'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.',
   'source': 'database_manual.txt'},
  0.17952925258413438)]

In [ ]:
# 파이썬 문법 함수를 만들 때
# 비슷한 동작을 하는 것들을 class : 변수, 메서드 하나로 묶는 단위(설계도)
class classname:
    def __init__():

In [17]:
# self : init에서 초기화 할 때 다른 함수들에서도 접근 할 수 있는 변수
class Calculator:
    def __init__(self):
        self.history = []

    def add(self, a, b):
        result = a+b
        self.history.append(f'{a} + {b} = {result}')
        return result

    def get_history(self):
        return self.history

In [18]:
calc = Calculator()

In [19]:
calc.history

[]

In [20]:
calc.result

AttributeError: 'Calculator' object has no attribute 'result'

In [21]:
calc.add(1, 2)

3

In [22]:
calc.history

['1 + 2 = 3']

In [23]:
calc.get_history()

['1 + 2 = 3']

In [67]:
class DocumentStore:
    def __init__(self):
        self.documents = []
        self.next_id = 1

    def add(self, content, source='unknown'):
        self.documents.append({
            'id' : self.next_id,
            'content' : content,
            'source' : source
        })
        self.next_id += 1

    def count(self):
        return len(self.documents)

    def search(self, keyword):
        return [doc for doc in self.documents if keyword in doc['content']]

    def retrieve(self, query, top_k=3):
        if not self.documents:
            return []
            
        contents = [doc['content'] for doc in self.documents]
        
        query_vec = embeddings.embed_query(query)
        doc_vecs = embeddings.embed_documents(contents)
        scores = cosine_similarity([query_vec], doc_vecs)[0]
        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return [(doc, float(score)) for doc, score in ranked][:top_k]
    
    def generate(self, query, top_k=3):
        # 1. 관련 문서 가져오기
        retrieved = self.retrieve(query, top_k=top_k)
    
        if not retrieved:
            return "관련 문서를 찾을 수 없습니다."
    
        documents = [doc['content'] for doc, score in retrieved]
        contexts = '\n'.join(f"- {doc}" for doc in documents)
    
        messages = [
            SystemMessage(content="제공된 문서를 기반으로 정확하게 답변해주세요."),
            HumanMessage(content=f"참고문서:\n{contexts}\n\n질문: {query}")
        ]
        response = llm.invoke(messages)
    
        return response.content

In [68]:
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색증강 생성이다', 'glossary.txt')

In [42]:
store.count()

2

In [29]:
store.search('파이썬')

[{'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'}]

In [54]:
store.retrieve('rag에 대해 설명해주세요')

[({'id': 2, 'content': 'RAG는 검색증강 생성이다', 'source': 'glossary.txt'},
  0.43624124667843533),
 ({'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'},
  0.2915541722268253)]

In [57]:
class WordCounter:
    def __init__(self):
        self.texts = []
        
    # 문자열을 저장
    def add_text(self, text):
        self.texts.append(text)

    # 현재까지 저장된 모든 문자열들의 단어 수
    def count_words(self):
        total = 0
        for text in self.texts:
            total += len(text.split())
        return total

In [58]:
wc = WordCounter()

In [61]:
wc.add_text('모든')

In [62]:
wc.count_words()

2

In [66]:
docs = ['rag는 검색증강생성 기술입니다.', '외부 문서를 검색해서 llm 답변에 활용합니다']
def rag_with_langchain(query, documents):
    if not documents:
        return "참고할 문서가 없습니다."
        
    contexts = '\n'.join(f"-{doc}" for doc in documents)
    messages=[
        SystemMessage(content='제공된 문서를 기반으로 정확하게 답변해주세요'),
        HumanMessage(content=f'참고문서:{contexts}\n\n질문:{query}')
    ]
    response = llm.invoke(messages)
    return response.content

In [65]:
rag_with_langchain('rag가 뭐야?', docs)

'RAG는 "Retrieval-Augmented Generation"의 약자로, 검색증강생성 기술입니다. 이 기술은 외부 문서나 데이터를 검색하여, 그 정보를 Large Language Model(LLM)의 답변 생성에 활용합니다. 즉, RAG는 모델이 정보를 찾고, 이를 기반으로 보다 정확하고 관련성 있는 답변을 생성할 수 있도록 도와줍니다.'

In [69]:
from langchain_core.documents import Document

In [70]:
doc = Document(page_content = "문서 내용...", metadata={"source" :"file.txt", "type":"polycy"})

In [71]:
file_path = SAMPLE_DIR / 'company_policy.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [72]:
raw_text

'주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'

In [73]:
doc = Document(page_content = raw_text, 
              metadata = {
                          "source" :file_path.name,
                          "file_type":"txt",
                          "char_count" : len(raw_text),
                          "line_count" : len(raw_text.splitlines())
                         })

In [74]:
doc

Document(metadata={'source': 'company_policy.txt', 'file_type': 'txt', 'char_count': 356, 'line_count': 19}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n')

In [76]:
# glob : 리스트로 조건에 대한 내용을 가져오도록 처리
def load_text_files(directory):
    documents = []
    for fp in sorted(directory.glob('*.txt')):
        text = fp.read_text(encoding='utf-8')
        doc = Document(page_content = raw_text, 
              metadata = {
                          "source" :fp.name,
                          "char_count" : len(text),
                         })
        documents.append(doc)

    return documents

In [77]:
all_docs = load_text_files(SAMPLE_DIR)

In [78]:
all_docs

[Document(metadata={'source': 'ai_report.txt', 'char_count': 396}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'),
 Document(metadata={'source': 'company_policy.txt', 'char_count': 356}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n연차휴가는 근로기준법에 따라 부여한다.\n경조사 휴가는 별도 규정에 따른다.\n자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n온라인 학습 플랫폼 이용료를 전액 지원한다.\n'),
 Document(metadata={'source': 'product_manual.txt', 'char_count': 329}, pag

In [83]:
def load_csv_file(file_path):
    documents = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            content = '|'.join(f'{k}:{v}' for k, v in row.items())
            doc = Document(page_content = content, 
              metadata = {
                          "source" :file_path.name,
                          "char_count" : len(content),
                         })
            documents.append(doc)
    return documents

In [84]:
csv_docs = load_csv_file(SAMPLE_DIR / 'employees.csv')
csv_docs

[Document(metadata={'source': 'employees.csv', 'char_count': 19}, page_content='이름:김철수|부서:개발팀|직급:선임'),
 Document(metadata={'source': 'employees.csv', 'char_count': 20}, page_content='이름:이영희|부서:기획팀|직급:매니저'),
 Document(metadata={'source': 'employees.csv', 'char_count': 19}, page_content='이름:박지민|부서:개발팀|직급:주임')]

In [85]:
test_json = {'name' : 'RAG 프로젝트', 'version' : '1.0', 'features' : ['검색', '생성']}

with open('sample_data/test.json', 'w', encoding='utf-8') as f:
    json.dump(test_json, f, ensure_ascii=False)

In [87]:
with open('sample_data/test.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

content = json.dumps(data, ensure_ascii=False, indent=2)

In [88]:
content

'{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}'

In [89]:
doc = Document(page_content = content, 
              metadata = {
                          "source" :'test.json',
                          "keys" : data.keys()
                         })

doc

Document(metadata={'source': 'test.json', 'keys': dict_keys(['name', 'version', 'features'])}, page_content='{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}')

In [90]:
def load_json_as_document(file_path):
    path = Path(file_path)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    content = json.dumps(data, ensure_ascii=False, indent=2)
    keys = list(data.keys())

    doc = Document(page_content = content, 
              metadata = {
                          "source" : path.name,
                          "keys" : keys
                         })

    return doc

In [ ]:
# 오늘은 전처리보다 파일에 대한 내용들 load 처리 하는 것들